In [9]:
!curl -o course_lead_scoring.csv https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80876  100 80876    0     0  80747      0  0:00:01  0:00:01 --:--:-- 80795


In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,precision_score,recall_score,f1_score
df = pd.read_csv('course_lead_scoring.csv')
cat_cols=df.select_dtypes(include='object').columns
df[cat_cols]=df[cat_cols].fillna('NA')
num_cols=df.select_dtypes(include=np.number).columns
df[num_cols]=df[num_cols].fillna(0.0)
df['converted']=(df['converted']==True).astype(int)
df_full_train,df_test=train_test_split(df,test_size=0.2,random_state=1)
df_train,df_val=train_test_split(df_full_train,test_size=0.25,random_state=1)
y_train=df_train['converted'].values
y_val=df_val['converted'].values
y_test=df_test['converted'].values
del df_train['converted']
del df_val['converted']
del df_test['converted']
dv=DictVectorizer(sparse=False)
X_train=dv.fit_transform(df_train.to_dict(orient='records'))
X_val=dv.transform(df_val.to_dict(orient='records'))
model=LogisticRegression(solver='lbfgs')
model.fit(X_train,y_train)
y_pred=model.predict_proba(X_val)[:,1]

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [20]:
numerical=['lead_score','number_of_courses_viewed','interaction_count','annual_income']
for f in numerical:
    auc=roc_auc_score(y_val,df_val[f])
    print(f'{f}:{auc:.3f}')

lead_score:0.623
number_of_courses_viewed:0.742
interaction_count:0.697
annual_income:0.554


In [21]:
auc=roc_auc_score(y_val,y_pred)
print(f'{auc:.3f}')

0.889


In [22]:
churn=(y_pred>=0.5)
precision=precision_score(y_val,churn)
recall=recall_score(y_val,churn)
print(precision,recall)

0.8064516129032258 0.8771929824561403


In [23]:
f1=f1_score(y_val,churn)
print(f1)

0.8403361344537815


In [42]:
from sklearn.model_selection import KFold
kfold=KFold(n_splits=5,shuffle=True,random_state=1)
scores=[]
for train_idx,val_idx in kfold.split(df_full_train):
    df_train_fold=df_full_train.iloc[train_idx]
    df_val_fold=df_full_train.iloc[train_idx]
    y_train_fold=df_train_fold['converted'].values
    y_val_fold=df_val_fold['converted'].values
    del df_train_fold['converted']
    del df_val_fold['converted']
    dv_fold=DictVectorizer(sparse=False)
    X_train_fold=dv_fold.fit_transform(df_train_fold.to_dict(orient='records'))
    X_val_fold=dv_fold.fit_transform(df_val_fold.to_dict(orient='records'))
    model_fold=LogisticRegression(solver='lbfgs',C=0.001,max_iter=1000)
    model_fold.fit(X_train_fold,y_train_fold)
    y_pred_fold=model_fold.predict_proba(X_val_fold)[:,1]
    scores.append(roc_auc_score(y_val_fold,y_pred_fold))
print(f'{np.std(scores):.3f}')

0.007


In [44]:
for c in [0.000001,0.001,1]:
    model_c=LogisticRegression(solver='lbfgs',C=c,max_iter=1000)
    model_c.fit(X_train,y_train)
    y_pread_c=model_c.predict_proba(X_val)[:,1]
    auc=roc_auc_score(y_val,y_pread_c)
    print(f"{c}{auc:.3f}")

1e-060.560
0.0010.848
10.921


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
